In [1]:
import jax
import jax.numpy as jnp
import netket as nk
import netket.experimental as nkx
import numpy as np
from pyscf import gto, scf, fci
from flax import linen as nn
import flax.nnx as nnx
import optax
from tqdm import tqdm
import time 
from functools import partial
from pyscf import gto, scf, fci
from jax import flatten_util
from itertools import combinations
from NES_VMC_H2_631G import get_ccsd_excitations_and_sampler_edges_from_hf,\
SingleStateAnsatz,create_machine,compute_local_energies,forces_expect_hermitian,compute_qgt


/opt/miniconda3/envs/Netket/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


∣NK⟩ Tip: You can disable these tips by setting export NETKET_NO_TIPS=1 in your .bashrc.

E0 = -1.02613572 Ha  |  激发能: 0.0000 eV
E1 = -0.97892204 Ha  |  激发能: 1.2848 eV
E2 = -0.66776157 Ha  |  激发能: 9.7519 eV
E3 = -0.60817046 Ha  |  激发能: 11.3734 eV


In [2]:
bond_length = 1.8
geometry = [('H', (0., 0., 0.)), ('H', (bond_length, 0., 0.))]
mol = gto.M(atom=geometry, basis='6-31G', verbose=0)
mf = scf.RHF(mol).run(verbose=0)
hf_ground_energy = mf.e_tot
print(f'HF 基准能量: {hf_ground_energy:.8f}')

cisolver = fci.FCI(mf)
cisolver.nroots = 4
E_fcis, fcivec = cisolver.kernel()
print("="*60)
print("H₂ FCI 基准能量")
print("="*60)
for i, e in enumerate(E_fcis):
    exc = (e - E_fcis[0]) * 27.2114
    print(f"E{i} = {e:.8f} Ha  |  激发能: {exc:.4f} eV")

# ha = nkx.operator.from_pyscf_molecule(mol)
hi = nk.hilbert.SpinOrbitalFermions(
    n_orbitals=4,
    s=1/2,
    n_fermions_per_spin=(1,1),
)
ha = nkx.operator.from_pyscf_molecule(mol)
Hatree_Fock = hi.all_states()[0]
sampler_edges, singles, doubles = get_ccsd_excitations_and_sampler_edges_from_hf(
    Hatree_Fock
)
print(f'sampler_edges: {sampler_edges}')
g = nk.graph.Graph(edges=sampler_edges)
single_rule = nk.sampler.rules.FermionHopRule(hilbert=hi, graph=g)
sampler = nk.sampler.MetropolisSampler(hi, rule=single_rule, n_chains=100, sweep_size=32)


HF 基准能量: -0.94605220
H₂ FCI 基准能量
E0 = -1.02613572 Ha  |  激发能: 0.0000 eV
E1 = -0.97892204 Ha  |  激发能: 1.2848 eV
E2 = -0.66776157 Ha  |  激发能: 9.7519 eV
E3 = -0.60817046 Ha  |  激发能: 11.3734 eV
sampler_edges: [(3, 0), (3, 1), (3, 2), (7, 4), (7, 5), (7, 6)]


In [3]:
# ===================== 6. 初始化 =====================
rngs = nnx.Rngs(21)
# 减小隐藏层维度，避免参数过多导致数值不稳定
model = SingleStateAnsatz(hi.size, hidden_dim=16, rngs=rngs)
machine, graphdef, params = create_machine(model)
sampler_state = sampler.init_state(machine, params, seed=1)

# 关键：自然梯度尺度远大于普通梯度，必须用极小的学习率
# 或者使用更稳定的优化器如 Adam
optimizer = optax.adam(learning_rate=0.001)  # 用 Adam 代替 SGD
# optimizer = optax.sgd(learning_rate=0.0001)  # 如果用 SGD，学习率要到 0.0001 甚至更小
opt_state = optimizer.init(params)

# 训练参数
N_ITER = 500  # 迭代次数
N_SAMPLES = 1008  # 样本数

# ===================== 7. 训练循环（带诊断指标）=====================
print("\n" + "="*90)
print("开始纯 JAX VMC 训练 (自然梯度下降法) - 带诊断指标")
print("="*90)
print(f"{'Step':>5} | {'E':>12} | {'±':>8} | {'Error':>8} | {'‖∇E‖':>10} | {'‖∇nat‖':>10} | {'cond(QGT)':>12} | {'‖params‖':>10}")
print("-"*90)

# 用于记录训练历史
history = {
    'step': [],
    'energy': [],
    'energy_std': [],
    'error': [],
    'grad_norm': [],
    'nat_grad_norm': [],
    'qgt_cond': [],
    'params_norm': []
}

for step in range(N_ITER):
    # 1. 采样
    sampler_state = sampler.reset(machine, params, sampler_state)
    
    samples, sampler_state = sampler.sample(
        machine, params, state=sampler_state, 
        chain_length=20
    )
    samples = samples.reshape(-1, hi.size)
    
    # 2. 计算 force-based 能量和梯度
    energy, energy_std, grad = forces_expect_hermitian(machine, params, samples)
    
    # 计算原始梯度范数（在乘以2之前）
    grad_flat_raw, _ = flatten_util.ravel_pytree(grad)
    grad_norm = float(jnp.sqrt(jnp.sum(jnp.abs(grad_flat_raw)**2)))
    
    # 梯度缩放因子
    grad = jax.tree_util.tree_map(lambda x: x * 2, grad)
    
    # 3. 计算 QGT（带诊断）
    qgt_reg, qgt_unravel_fun = compute_qgt(machine, params, samples, diag_shift=0.1)
    
    # 计算 QGT 条件数（使用 SVD 的最大/最小奇异值比）
    qgt_for_svd = jnp.real(qgt_reg)  # 取实部做 SVD
    s = jnp.linalg.svd(qgt_for_svd, compute_uv=False)
    qgt_cond = float(s[0] / (s[-1] + 1e-10))  # 避免除以零
    
    grad_flat, grad_unravel_fn = flatten_util.ravel_pytree(grad)
  
    # 4. 自然梯度 natural-gradient = S^{-1} * grad（使用更稳定的 solve）
    try:
        natural_grad_flat = jnp.linalg.solve(qgt_reg, grad_flat)
        natural_grad = grad_unravel_fn(natural_grad_flat)
        nat_grad_norm = float(jnp.sqrt(jnp.sum(jnp.abs(natural_grad_flat)**2)))
    except Exception as e:
        # 如果 solve 失败，使用 SVD 求逆作为后备
        print(f"  [WARNING] solve failed at step {step}: {e}, using SVD fallback")
        U, s_svd, Vh = jnp.linalg.svd(qgt_reg, full_matrices=False)
        sinv = 1.0 / (s_svd + 0.01)  # 加正则化
        natural_grad_flat = Vh.T @ (sinv[:, None] * (U.T @ grad_flat))
        natural_grad = grad_unravel_fn(natural_grad_flat)
        nat_grad_norm = float(jnp.sqrt(jnp.sum(jnp.abs(natural_grad_flat)**2)))
    
    grad = natural_grad
    
    # 5. 更新参数（自然梯度下降）
    updates, opt_state = optimizer.update(grad, opt_state, params)
    params = optax.apply_updates(params, updates)
    
    # 6. 计算参数范数
    params_flat, _ = flatten_util.ravel_pytree(params)
    params_norm = float(jnp.sqrt(jnp.sum(jnp.abs(params_flat)**2)))
    
    # 7. 记录历史
    if step % 50 == 0 or step == N_ITER - 1:
        error = jnp.abs(energy.real - E_fcis[0])
        history['step'].append(step)
        history['energy'].append(float(energy.real))
        history['energy_std'].append(float(energy_std))
        history['error'].append(float(error))
        history['grad_norm'].append(grad_norm)
        history['nat_grad_norm'].append(nat_grad_norm)
        history['qgt_cond'].append(qgt_cond)
        history['params_norm'].append(params_norm)
        
        # 格式化输出
        E_str = f"{energy.real:.8f}" if not jnp.isnan(energy.real) else "nan"
        std_str = f"{energy_std:.6f}" if not jnp.isnan(energy_std) else "nan"
        err_str = f"{float(error):.6f}" if not jnp.isnan(error) else "nan"
        gn_str = f"{grad_norm:.6f}"
        ngn_str = f"{nat_grad_norm:.6f}"
        cond_str = f"{qgt_cond:.2e}"
        pn_str = f"{params_norm:.4f}"
        
        print(f"{step:>5} | {E_str:>12} | {std_str:>8} | {err_str:>8} | {gn_str:>10} | {ngn_str:>10} | {cond_str:>12} | {pn_str:>10}")

# 最终结果
final_energy, final_std, _ = forces_expect_hermitian(machine, params, samples)
final_error = jnp.abs(final_energy.real - E_fcis[0])
print("\n" + "="*90)
print(f"训练完成!")
print(f"最终能量：{final_energy.real:.8f} ± {final_std:.6f} Ha")
print(f"FCI 基准：{E_fcis[0]:.8f} Ha")
print(f"绝对误差：{final_error:.6f} Ha")
print(f"相对误差：{final_error / jnp.abs(E_fcis[0]) * 100:.4f}%")
print("="*90)

# 打印诊断摘要
print("\n" + "="*90)
print("诊断摘要")
print("="*90)
print(f"QGT 条件数范围: {min(history['qgt_cond']):.2e} ~ {max(history['qgt_cond']):.2e}")
print(f"梯度范数范围:   {min(history['grad_norm']):.6f} ~ {max(history['grad_norm']):.6f}")
print(f"自然梯度范数:   {min(history['nat_grad_norm']):.6f} ~ {max(history['nat_grad_norm']):.6f}")
print(f"参数范数范围:   {min(history['params_norm']):.4f} ~ {max(history['params_norm']):.4f}")
print("="*90)



开始纯 JAX VMC 训练 (自然梯度下降法) - 带诊断指标
 Step |            E |        ± |    Error |       ‖∇E‖ |     ‖∇nat‖ |    cond(QGT) |   ‖params‖
------------------------------------------------------------------------------------------
    0 |   0.54212596 | 0.018589 | 1.568262 |   0.892734 |  13.001828 |     1.19e+01 |     5.7731
   50 |  -0.59664129 | 0.010450 | 0.429494 |   0.519526 |   4.828017 |     4.12e+01 |     5.9138
  100 |  -0.96332882 | 0.001965 | 0.062807 |   0.057031 |   0.445444 |     8.24e+03 |     5.9411
  150 |  -0.97297518 | 0.002617 | 0.053161 |   0.371964 |   1.550148 |     3.96e+03 |     5.9443
  200 |  -0.98366862 | 0.004362 | 0.042467 |   0.375383 |   1.236720 |     2.10e+03 |     5.9461
  250 |  -0.99017872 | 0.003779 | 0.035957 |   0.230039 |   0.634452 |     2.79e+03 |     5.9500
  300 |  -0.99608850 | 0.003010 | 0.030047 |   0.477498 |   1.341361 |     4.30e+03 |     5.9514
  350 |  -1.00646975 | 0.002637 | 0.019666 |   2.034394 |   7.588762 |     5.16e+03 |     5.9616
  

In [4]:
class SingleStateAnsatz(nnx.Module):
    def __init__(self, n_spin_orbitals: int, hidden_dim=16, *, rngs: nnx.Rngs):
        super().__init__()
        self.linear1 = nnx.Linear(n_spin_orbitals, hidden_dim, rngs=rngs, param_dtype=complex)
        self.linear2 = nnx.Linear(hidden_dim, hidden_dim, rngs=rngs, param_dtype=complex)
        self.output = nnx.Linear(hidden_dim, 1, rngs=rngs, param_dtype=complex)

    def __call__(self, x):
        h = nnx.tanh(self.linear1(x.astype(complex)))
        h = nnx.tanh(self.linear2(h))
        out = self.output(h)
        return jnp.squeeze(out)

def create_machine(model: nnx.Module):
    """将 Flax NNX 模型包装为 NetKet 风格的 machine 函数"""
    graphdef, state = nnx.split(model)
    
    @jax.jit
    def machine(params, sigma):
        m = nnx.merge(graphdef, params)
        return m(sigma)
    
    return machine, graphdef, state

# ===================== 5. 纯 JAX 实现的 force-based 梯度计算 =====================
@partial(jax.jit, static_argnames=("machine",))
def compute_local_energies(machine, params, sigma):
    """
    计算局部能量 E_loc(σ) = Σ_η H(σ→η) ψ(η)/ψ(σ)
    
    这对应 NetKet 的 local_value_kernel
    """
    eta, H_eta = ha.get_conn_padded(sigma)
    logpsi_sigma = machine(params, sigma)
    logpsi_eta = machine(params, eta)
    logpsi_sigma = jnp.expand_dims(logpsi_sigma, -1)
    return jnp.sum(H_eta * jnp.exp(logpsi_eta - logpsi_sigma), axis=-1)

def statistics(x):
    """计算样本统计量"""
    mean = jnp.mean(x)
    var = jnp.var(x)
    return mean, jnp.sqrt(var / x.shape[0])

@partial(jax.jit, static_argnames=("machine",))
def forces_expect_hermitian(machine, params, sigma):
    """
    核心：复刻 NetKet 的 forces_expect_hermitian 函数
    
    使用 force-based 梯度计算：
    ∇⟨E⟩ = ⟨(E_loc - ⟨E⟩) ∇log ψ⟩
    
    关键：对于复数值网络，使用 holomorphic=True
    """
    # 1. 计算局部能量
    O_loc = compute_local_energies(machine, params, sigma)
    
    # 2. 统计能量均值
    O_mean, O_std = statistics(O_loc)
    
    # 3. 中心化局部能量
    O_centered = O_loc - O_mean
    
    # 4. 计算 ∇log ψ 对每个样本
    # 使用 jax.grad 计算复数梯度（holomorphic=True）
    def log_psi_single(p, s):
        return machine(p, s)
    
    def compute_grad_for_sample(s):
        return jax.grad(lambda p: log_psi_single(p, s), holomorphic=True)(params)
    
    grad_matrix = jax.vmap(compute_grad_for_sample)(sigma)
    
    # 5. 计算 force-based 梯度
    # grad = ⟨(E_loc - E_mean) ∇log ψ⟩ = (1/N) Σ (E_loc[i] - E_mean) ∇log ψ(σ[i])
    # grad_matrix 已经是 PyTree 结构，每个元素的形状是 (n_samples, ...)
    # 关键修复：O_centered 形状为 (n_samples,)，需要正确广播到梯度张量的每个维度
    # 使用 reshape 将 O_centered 变为 (n_samples, 1, 1, ..., 1) 以匹配梯度张量
    def weight_and_mean(grad_component):
        # grad_component 形状：(n_samples, d1, d2, ...)
        # O_centered 形状：(n_samples,)
        # 需要广播相乘后沿 axis=0 求平均
        weights = O_centered.reshape((O_centered.shape[0],) + (1,) * (grad_component.ndim - 1))
        return jnp.mean(weights * jnp.conj(grad_component), axis=0)
    
    grad = jax.tree_util.tree_map(weight_and_mean, grad_matrix)
    
    return O_mean, O_std, grad


def get_ccsd_excitations_and_sampler_edges_from_hf(hf_state):
    """
    返回：
    1. sampler_edges: 给 NetKet Graph 用，格式为 [(i, a), ...]
    2. singles:       物理 single excitation，格式为 [((i, a),), ...]
    3. doubles:       物理 double excitation，格式为 [((i,a), (j,b)), ...]
    """
    hf_state = np.asarray(hf_state)
    n_spin_orbitals = hf_state.size
    assert n_spin_orbitals % 2 == 0

    nmo = n_spin_orbitals // 2

    sampler_edges = []

    # alpha block: 0 ~ nmo-1
    alpha_sites = np.arange(0, nmo)
    alpha_occ = alpha_sites[hf_state[alpha_sites] == 1]
    alpha_vir = alpha_sites[hf_state[alpha_sites] == 0]

    for i in alpha_occ:
        for a in alpha_vir:
            sampler_edges.append((int(i), int(a)))

    # beta block: nmo ~ 2*nmo-1
    beta_sites = np.arange(nmo, 2 * nmo)
    beta_occ = beta_sites[hf_state[beta_sites] == 1]
    beta_vir = beta_sites[hf_state[beta_sites] == 0]

    for i in beta_occ:
        for a in beta_vir:
            sampler_edges.append((int(i), int(a)))

    # 物理意义上的 single excitation
    singles = [(edge,) for edge in sampler_edges]

    # 物理意义上的 double excitation
    doubles = []
    for move1, move2 in combinations(sampler_edges, 2):
        i, a = move1
        j, b = move2

        # 不能动同一个电子，也不能占到同一个虚轨道
        if i != j and a != b:
            doubles.append((move1, move2))

    return sampler_edges, singles, doubles

In [5]:
sampler_edges, singles, doubles = get_ccsd_excitations_and_sampler_edges_from_hf(
    Hatree_Fock
)
sampler_edges

[(3, 0), (3, 1), (3, 2), (7, 4), (7, 5), (7, 6)]

In [6]:
# ======================
# 超参数
# ======================
N_CHAINS = 16
N_WARMUP = 32
N_SAMPLES_PER_CHAIN = 100
SWEEP_SIZE = 32

# ======================
# 初始化 ONCE
# ======================
rngs = nnx.Rngs(21)
model = SingleStateAnsatz(4, hidden_dim=12, rngs=rngs)
machine, graphdef, params = create_machine(model)

sampler_state = init_sampler_state(hi, N_CHAINS, seed=42)
samples, sampler_state = mcmc_sampler_multichain(
    n_samples_per_chain=N_SAMPLES_PER_CHAIN,
    n_warmup=N_WARMUP,
    sampler_state=sampler_state,  # ✅ 状态传递
    edges=((0,1),(2,3)),
    machine=machine,
    params=params,
    sweep_size=SWEEP_SIZE
)
samples.shape

NameError: name 'init_sampler_state' is not defined

In [ ]:
import jax
import jax.numpy as jnp
import optax
import time
from functools import partial
import flax.nnx as nnx

# ===================== 6. 初始化（适配多链） =====================
rngs = nnx.Rngs(21)
model = SingleStateAnsatz(4, hidden_dim=12, rngs=rngs)
machine, graphdef, params = create_machine(model)

optimizer = optax.sgd(learning_rate=0.01)
opt_state = optimizer.init(params)

# 推荐参数（和 NetKet 一样快、一样准）
N_CHAINS = 100
N_SAMPLES_PER_CHAIN = 20
N_WARMUP = 10
SWEEP_SIZE = 32
N_ITER = 300

# ===================== ✅ 关键：初始化 sampler_state（只初始化一次！）=====================
def init_sampler_state(hi, n_chains, seed=42):
    init_states = generate_random_initial_states(hi, n_chains, seed)
    key = jax.random.PRNGKey(seed)
    chain_keys = jax.random.split(key, n_chains)
    return (init_states, chain_keys)

# 初始化一次，后面永远复用、更新
sampler_state = init_sampler_state(hi, N_CHAINS, seed=21)

# ===================== 7. 训练循环（✅ 完全修复版）=====================
print("\n" + "="*60)
print("开始多链 VMC 训练 (自然梯度下降法)")
print("="*60)

history = {
    'step': [],
    'energy': [],
    'energy_std': [],
    'error': []
}
start_time = time.time()

for step in range(N_ITER):
    # ======================================================================
    # ✅ 1. 采样：复用 sampler_state，不每次重新生成初始态！（核心修复）
    # ======================================================================
    samples, sampler_state = mcmc_sampler_multichain(
        n_samples_per_chain=N_SAMPLES_PER_CHAIN,
        n_warmup=N_WARMUP,
        sampler_state=sampler_state,  # 状态传递
        edges=((0,1),(2,3)),
        machine=machine,
        params=params,
        sweep_size=SWEEP_SIZE
    )

    # ======================================================================
    # 2. 能量 & 自然梯度（不变）
    # ======================================================================
    energy, energy_std, grad = forces_expect_hermitian(machine, params, samples)
    grad = jax.tree_map(lambda x: x * 2, grad)

    qgt_reg, qgt_unravel_fun = compute_qgt(machine, params, samples, diag_shift=0.001)
    grad_flat, grad_unravel_fn = flatten_util.ravel_pytree(grad)
    natural_grad_flat = jnp.linalg.solve(qgt_reg, grad_flat)
    natural_grad = grad_unravel_fn(natural_grad_flat)

    # ======================================================================
    # 3. 参数更新
    # ======================================================================
    updates, opt_state = optimizer.update(natural_grad, opt_state, params)
    params = optax.apply_updates(params, updates)

    # ======================================================================
    # 4. 日志
    # ======================================================================
    if step % 50 == 0 or step == N_ITER - 1:
        error = jnp.abs(energy.real - E_fcis[0])
        history['step'].append(step)
        history['energy'].append(float(energy.real))
        history['energy_std'].append(float(energy_std))
        history['error'].append(float(error))
        print(f"Step {step:3d} | E: {energy.real:.8f} ± {energy_std:.6f} | FCI: {E_fcis[0]:.8f} | Error: {error:.6f}")

end_time = time.time()
print(f"\n训练耗时：{end_time - start_time:.2f} 秒")

# ======================================================================
# 最终能量评估
# ======================================================================
final_samples, _ = mcmc_sampler_multichain(
    n_samples_per_chain=N_SAMPLES_PER_CHAIN * 2,
    n_warmup=5,
    sampler_state=sampler_state,
    edges=((0,1),(2,3)),
    machine=machine,
    params=params,
    sweep_size=SWEEP_SIZE
)
final_energy, final_std, _ = forces_expect_hermitian(machine, params, final_samples)
final_error = jnp.abs(final_energy.real - E_fcis[0])

print("\n" + "="*60)
print(f"训练完成!")
print(f"最终能量：{final_energy.real:.8f} ± {final_std:.6f} Ha")
print(f"FCI 基准：{E_fcis[0]:.8f} Ha")
print(f"绝对误差：{final_error:.6f} Ha")
print(f"相对误差：{final_error / jnp.abs(E_fcis[0]) * 100:.4f}%")
print("="*60)

In [ ]:
jax.__version__